# Visible ships — modern models

Run setup cells in order, then one model at a time. Shared seed 42, 1,000 train / 250 validation / 500 test; 500 maximum epochs, patience 20. Patience and best-checkpoint selection use validation mask AP50:95. Test is evaluated after training. SAMSAR uses ground-truth box prompts and is labelled accordingly.


## 1. Mount Drive and extract the files


In [3]:
from pathlib import Path
from google.colab import drive
import json, os, sys, subprocess, zipfile

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
ROOT = DRIVE / 'Visible_Ship_Study'
# Preserve existing code, configurations, checkpoints and completed runs.
if not (ROOT / 'scripts/setup_environment.py').is_file():
    package = DRIVE / 'Visible_Ship_Study_Package.zip'
    assert package.is_file(), f'Upload {package.name} to MyDrive'
    with zipfile.ZipFile(package) as archive:
        assert archive.testzip() is None, 'Package ZIP is damaged'
        archive.extractall(DRIVE)
manifest = ROOT / 'data/visible/manifest.json'
if not manifest.is_file():
    with zipfile.ZipFile(DRIVE / 'Visible_Ship_Dataset.zip') as archive:
        assert archive.testzip() is None, 'Dataset ZIP is damaged'
        archive.extractall(DRIVE)
dataset = json.loads(manifest.read_text())
assert dataset['seed'] == 42
assert dataset['counts'] == {'train': 1000, 'val': 250, 'test': 500}
print('Dataset:', dataset['counts'])
print('Project:', ROOT)


Mounted at /content/drive
Dataset: {'train': 1000, 'val': 250, 'test': 500}
Project: /content/drive/MyDrive/Visible_Ship_Study


## 2. GPU and environment

Use A100 or T4. MMDetection requires Python 3.11 and cannot use the Blackwell G4 GPU with these packages. Modern models can use Python 3.11–3.13.


In [3]:
!nvidia-smi


Thu Oct  8 10:51:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
subprocess.run([sys.executable, str(ROOT / 'scripts/setup_environment.py'), 'modern'], check=True)
PYTHON = Path('/content/visible-modern-env/bin/python')
subprocess.run([str(PYTHON), '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))'], check=True)


CompletedProcess(args=['/content/visible-modern-env/bin/python', '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))'], returncode=0)

## 3. ClearML credentials

Set CLEARML_ACCESS_KEY and CLEARML_SECRET_KEY in Colab Secrets.


In [5]:
from google.colab import userdata
os.environ['CLEARML_API_ACCESS_KEY'] = userdata.get('CLEARML_ACCESS_KEY')
os.environ['CLEARML_API_SECRET_KEY'] = userdata.get('CLEARML_SECRET_KEY')
os.environ['CLEARML_API_HOST'] = 'https://api.clear.ml'
os.environ['CLEARML_WEB_HOST'] = 'https://app.clear.ml'
os.environ['CLEARML_FILES_HOST'] = 'https://files.clear.ml'
assert os.environ['CLEARML_API_ACCESS_KEY'] and os.environ['CLEARML_API_SECRET_KEY']
print('ClearML project: Visible Ship Segmentation Study')


ClearML project: Visible Ship Segmentation Study


## 4. Verify the shared settings


In [6]:
settings = json.loads((ROOT / 'configs/visible.json').read_text())
print('Epoch ceiling:', settings['max_epochs'], '| Patience:', settings['patience'])
print('Seed:', settings['seed'], '| Batch:', settings['batch_size'], '| AMP:', settings['amp'])
print('Split sizes:', dataset['counts'])
assert settings['seed'] == 42 and settings['max_epochs'] == 500 and settings['patience'] == 20


Epoch ceiling: 500 | Patience: 20
Seed: 42 | Batch: 1 | AMP: False
Split sizes: {'train': 1000, 'val': 250, 'test': 500}


## 5. Run helper


In [7]:
def run_one(model, smoke=True):
    config = ROOT / 'configs' / f'{model}.json'
    command = [str(PYTHON), '-u', '-m', 'sarstudy.runner', '--config', str(config),
               '--model', model, '--dataset', 'visible']
    if smoke:
        command.append('--smoke')
    environment = os.environ.copy()
    environment['MPLBACKEND'] = 'Agg'
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, env=environment)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    if return_code:
        raise RuntimeError(f'{model}/visible failed: exit code {return_code}; see preceding error')
    print('Completed:', model)


## SAMSAR prerequisites

Run this before the SAMSAR cells. Your existing SAMSAR implementation folder is required.


In [8]:
import urllib.request
repo = Path(settings['samsar_repo'])
assert (repo / 'samsar/modeling/build_samsar.py').is_file(), f'SAMSAR implementation missing: {repo}'
checkpoint = Path(settings['sam_checkpoint'])
if not checkpoint.is_file():
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    temporary = checkpoint.with_suffix('.download')
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth', temporary)
    assert temporary.stat().st_size > 100_000_000
    temporary.replace(checkpoint)
print('SAMSAR prerequisites ready')


SAMSAR prerequisites ready


## 6. Two-epoch smoke tests

These check execution, saved loss history and validation logging. Low AP after two epochs does not measure final performance. Run each model cell once.


### yolo26n_seg


In [ ]:
run_one('yolo26n_seg', smoke=True)


ClearML Task: created new task id=978669f9255a4ab884ceada1e90407ba
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/978669f9255a4ab884ceada1e90407ba/output/log
{'train': 8, 'val': 4, 'test': 4}
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.7.1+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, co

### yolo26s_seg


In [ ]:
run_one('yolo26s_seg', smoke=True)


ClearML Task: created new task id=abe77a5c6f4f4ea29b56c26031235de9
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/abe77a5c6f4f4ea29b56c26031235de9/output/log
{'train': 8, 'val': 4, 'test': 4}

New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.7.1+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/visible_ship_cache/4c50b730d382e3e1991880b9b1d532f2254bc3e6d6d046238e829a8e3d8bf39c/visible/data.yaml, degrees=0.0, deterministic=False, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=Fals

### yolo26m_seg


In [ ]:
run_one('yolo26m_seg', smoke=True)


ClearML Task: created new task id=66e78e465436442b9fdda6dcf5e82a3d
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/66e78e465436442b9fdda6dcf5e82a3d/output/log
{'train': 8, 'val': 4, 'test': 4}

New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.7.1+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/visible_ship_cache/4c50b730d382e3e1991880b9b1d532f2254bc3e6d6d046238e829a8e3d8bf39c/visible/data.yaml, degrees=0.0, deterministic=False, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=Fals

### rfdetr_seg


In [ ]:
run_one('rfdetr_seg', smoke=True)


ClearML Task: created new task id=510bce3b36f74cb3850477a416622179
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/510bce3b36f74cb3850477a416622179/output/log
{'train': 8, 'val': 4, 'test': 4}
[2026-10-07 08:05:16] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-07 08:05:16] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-07 08:05:17] [INFO] rf-detr - Downloading pretrained weights for rf-detr-seg-nano.pt

rf-detr-seg-nano.pt:   0%|          | 0.00/128M [00:00<?, ?iB/s]
rf-detr-seg-nano.pt:   0%|          | 47.0k/128M [00:00<12:05, 185kiB/s]
rf-detr-seg-nano.pt:   0%|          | 79.0k/128M [00:00<13:49, 162kiB/s]
rf-detr-seg-nano.pt:   0%|          | 1

### samsar


In [ ]:
run_one('samsar', smoke=True)


ClearML Task: created new task id=586546739eb44f69802b2053a41ed2a8
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/586546739eb44f69802b2053a41ed2a8/output/log
{'train': 8, 'val': 4, 'test': 4}
[SAMSAR] loaded 314/314 pre-trained SAM tensors (114 new-module tensors initialized from scratch, 0 checkpoint tensors unused)
[SAMSAR] parameters: 132.0M total, 42.3M trainable
2026-10-07 08:13:29,335 - clearml.Task - ERROR - Action failed <400/110: tasks.add_or_update_artifacts/v2.10 (Invalid task status: expected=(created, queued, in_progress), status=failed)> (task=586546739eb44f69802b2053a41ed2a8, artifacts=[{'key': 'split_manifest', 'type': 'custom', 'uri': 'https://files.clear.ml/Visible%20Ship%20Segmentation%20Study/visible_v1_seed42_smoke%20%257C%20visible%20%257C%20samsar%20%257C%20attempt%2002.586546739eb44f69802b2053a41ed2a8/artifacts/split_manifest/manifest.json', 'content_size': 113261, 'hash': '3583540199e990c70617ae0a180c058ebff8b83b7df0d

RuntimeError: samsar/visible failed: exit code -9; see preceding error

In [ ]:
# SAMSAR uses its reference implementation's fixed 256x256 input.
# Only SAMSAR gets a new run ID; existing model runs and splits are preserved.
import json
p = ROOT / 'configs/samsar.json'
cfg = json.loads(p.read_text())
backup = p.with_suffix('.before_size_fix.json')
if not backup.exists():
    backup.write_text(p.read_text())
cfg['samsar_img_size'] = 256
cfg['experiment'] = 'visible_v2_samsar256_seed42'
p.write_text(json.dumps(cfg, indent=2))
print('SAMSAR input:', cfg['samsar_img_size'], '| Seed:', cfg['seed'])
print('Experiment:', cfg['experiment'])
run_one('samsar', smoke=True)


SAMSAR input: 256 | Seed: 42
Experiment: visible_v2_samsar256_seed42
ClearML Task: created new task id=384e7fc41ecd4f3e98d1dd7a6a8b4cf2
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/384e7fc41ecd4f3e98d1dd7a6a8b4cf2/output/log
{'train': 8, 'val': 4, 'test': 4}
[SAMSAR] loaded 314/314 pre-trained SAM tensors (114 new-module tensors initialized from scratch, 0 checkpoint tensors unused)
[SAMSAR] parameters: 132.0M total, 42.3M trainable
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=0.00s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.437
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.752
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxD

### ship_contour


In [ ]:
run_one('ship_contour', smoke=True)


ClearML Task: created new task id=47af31ac23c14068a4538471c18db605
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/47af31ac23c14068a4538471c18db605/output/log
{'train': 8, 'val': 4, 'test': 4}
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=0.02s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=1

## 7. Full training

After checking the smoke output, set the following value to True. On a new session, rerun setup and the helper, then rerun the relevant full-training cell to resume. Do not change settings during a run.


In [10]:
APPROVE_FULL_TRAINING = True


### yolo26n_seg


In [ ]:
assert APPROVE_FULL_TRAINING, 'Set APPROVE_FULL_TRAINING = True after the smoke check'
run_one('yolo26n_seg', smoke=False)


Streaming output truncated to the last 5000 lines.
     86/500      2.34G     0.8583      1.455     0.9081    0.01407     0.7701          2        512: 43% ━━━━━─────── 435/1000 8.7it/s 50.4s<1:05
     86/500      2.34G     0.8577      1.455      0.909    0.01406     0.7697          6        512: 43% ━━━━━─────── 436/1000 8.8it/s 50.5s<1:04
     86/500      2.34G     0.8598      1.458     0.9143    0.01406     0.7719          7        512: 43% ━━━━━─────── 437/1000 8.7it/s 50.6s<1:04
     86/500      2.34G     0.8603      1.457     0.9146    0.01405     0.7716          5        512: 43% ━━━━━─────── 438/1000 8.7it/s 50.8s<1:04
     86/500      2.34G     0.8607      1.457     0.9145    0.01404     0.7713          7        512: 43% ━━━━━─────── 439/1000 8.7it/s 50.9s<1:04
     86/500      2.34G     0.8613       1.46     0.9141    0.01407     0.7773          3        512: 44% ━━━━━─────── 440/1000 8.8it/s 51.0s<1:04
     86/500      2.34G     0.8616       1.46     0.9158    0.01411     0.

### yolo26s_seg


In [23]:
assert APPROVE_FULL_TRAINING, 'Set APPROVE_FULL_TRAINING = True after the smoke check'
run_one('yolo26s_seg', smoke=False)


Streaming output truncated to the last 5000 lines.
     80/500     0.953G      0.721      1.202     0.5703    0.01149     0.5449         11        512: 52% ━━━━━━────── 525/1000 8.7it/s 58.8s<54.8s
     80/500     0.953G     0.7215      1.203     0.5701     0.0115     0.5464          1        512: 52% ━━━━━━────── 526/1000 8.7it/s 58.9s<54.5s
     80/500     0.953G     0.7211      1.202     0.5693     0.0115     0.5459          3        512: 52% ━━━━━━────── 527/1000 8.7it/s 59.0s<54.5s
     80/500     0.953G     0.7204      1.202     0.5686    0.01149     0.5454          2        512: 52% ━━━━━━────── 528/1000 8.7it/s 59.1s<54.1s
     80/500     0.953G     0.7199      1.201     0.5682    0.01148      0.545          2        512: 52% ━━━━━━────── 529/1000 8.7it/s 59.2s<54.0s
     80/500     0.953G     0.7177      1.198     0.5677    0.01145     0.5438          2        512: 53% ━━━━━━────── 531/1000 9.2it/s 59.4s<51.2s
     80/500     0.953G     0.7174      1.198     0.5671    0.01144 

### yolo26m_seg


In [ ]:
assert APPROVE_FULL_TRAINING, 'Set APPROVE_FULL_TRAINING = True after the smoke check'
run_one('yolo26m_seg', smoke=False)


Streaming output truncated to the last 5000 lines.
      3/500      1.62G      1.335      2.489      1.462    0.02445      1.395          1        512: 1% ──────────── 14/1000 8.1it/s 1.9s<2:02
      3/500      1.62G      1.393      2.518      1.519    0.02809      1.636          2        512: 1% ──────────── 15/1000 8.2it/s 2.0s<2:01
      3/500      1.62G      1.363      2.454      1.497    0.02712      1.578          3        512: 1% ──────────── 16/1000 8.2it/s 2.1s<2:01
      3/500      1.62G      1.345      2.421      1.512     0.0264      1.642          5        512: 1% ──────────── 17/1000 8.0it/s 2.2s<2:02
      3/500      1.62G       1.36      2.379      1.466    0.02683      1.609          3        512: 1% ──────────── 18/1000 8.0it/s 2.4s<2:03
      3/500      1.62G      1.347      2.318      1.415    0.02639       1.57          4        512: 1% ──────────── 19/1000 8.1it/s 2.5s<2:01
      3/500      1.62G      1.339      2.282      1.411    0.02772      1.547          1   

### rfdetr_seg


In [9]:
assert APPROVE_FULL_TRAINING, 'Set APPROVE_FULL_TRAINING = True after the smoke check'
run_one('rfdetr_seg', smoke=False)


ClearML Task: created new task id=1db2a0a96eac42f2bb416d18cd5d1a30
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/1db2a0a96eac42f2bb416d18cd5d1a30/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
{'train': 1000, 'val': 250, 'test': 500}

                                            0% | 0.00/11.8 MB [00:00<?, ?MB/s]: 
███████████████████████████       85% | 10.00/11.8 MB [00:00<00:00, 21.13MB/s]: 
████████████████████████████████ 100% | 11.80/11.8 MB [00:01<00:00, 10.65MB/s]: 
[2026-10-08 05:12:11] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-08 05:12:11] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DE

### samsar


In [11]:
assert APPROVE_FULL_TRAINING, 'Set APPROVE_FULL_TRAINING = True after the smoke check'
run_one('samsar', smoke=False)


ClearML Task: created new task id=d53a9b31be0340cf97e1b28ac0f528f3
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/d53a9b31be0340cf97e1b28ac0f528f3/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
{'train': 1000, 'val': 250, 'test': 500}

                                            0% | 0.00/11.8 MB [00:00<?, ?MB/s]: 
███████████████████████████       85% | 10.00/11.8 MB [00:00<00:00, 21.73MB/s]: 
████████████████████████████████ 100% | 11.80/11.8 MB [00:01<00:00, 10.93MB/s]: 
[SAMSAR] loaded 314/314 pre-trained SAM tensors (114 new-module tensors initialized from scratch, 0 checkpoint tensors unused)
[SAMSAR] parameters: 132.0M total, 42.3M trainable
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DON

### ship_contour


In [12]:
assert APPROVE_FULL_TRAINING, 'Set APPROVE_FULL_TRAINING = True after the smoke check'
run_one('ship_contour', smoke=False)


ClearML Task: created new task id=e2144d2cd00c41dab5bdbb576a713e87
ClearML results page: https://app.clear.ml/projects/12708556914046aabb9ea7b23aa450fb/tasks/e2144d2cd00c41dab5bdbb576a713e87/output/log
{'train': 1000, 'val': 250, 'test': 500}

                                            0% | 0.00/11.8 MB [00:00<?, ?MB/s]: 
███████████████████████████       85% | 10.00/11.8 MB [00:00<00:00, 21.83MB/s]: 
████████████████████████████████ 100% | 11.80/11.8 MB [00:01<00:00, 10.84MB/s]: 
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.33s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=1.47s).
Accumulating evaluation results...
DONE (t=0.17s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | 

## 8. Confirm completed outputs


In [4]:
# Read saved completion markers only; no training or evaluation is rerun.
from pathlib import Path
import json

ROOT = Path('/content/drive/MyDrive/Visible_Ship_Study')
models = ['yolo26n_seg', 'yolo26s_seg', 'yolo26m_seg',
          'rfdetr_seg', 'samsar', 'ship_contour']
all_complete = True

for model in models:
    config_path = ROOT / 'configs' / f'{model}.json'
    if not config_path.is_file():
        print(model, 'configuration missing:', config_path)
        all_complete = False
        continue
    model_settings = json.loads(config_path.read_text())
    folder = ROOT / 'runs' / 'visible' / model / model_settings['experiment']
    trained = (folder / 'training_complete.json').is_file()
    evaluated = (folder / 'evaluation_complete.json').is_file()
    print(f'{model:15} training: {trained} | evaluation: {evaluated}')
    all_complete = all_complete and trained and evaluated

print()
print('All six training and evaluation runs complete:', all_complete)


yolo26n_seg     training: True | evaluation: True
yolo26s_seg     training: True | evaluation: True
yolo26m_seg     training: True | evaluation: True
rfdetr_seg      training: True | evaluation: True
samsar          training: True | evaluation: True
ship_contour    training: True | evaluation: True

All six training and evaluation runs complete: True
